# montecarlo-hermes quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tetttet/montecarlo-hermes/blob/main/examples/quickstart.ipynb)

Simulate a decision, read the risk, find the drivers and see what is worth learning first.

In [ ]:
%pip install -q montecarlo-hermes

In [ ]:
import numpy as np

import mchermes as mc

d = mc.Decision("Open a second coffee shop?", n=100_000, seed=42)
rent = d.uncertain("rent", mc.Normal(3000, 300))
sales = d.uncertain("monthly_sales", mc.Triangular(8000, 12000, 18000))
margin = d.uncertain("margin", mc.Uniform(0.25, 0.4))

d.option("open", lambda: sales * margin * 12 - rent * 12 - 20_000)
d.option("dont_open", 0)

r = d.run()
r  # a Result renders as a table in notebooks

## Distributions and drivers

In [ ]:
r.plot();

In [ ]:
r.plot_tornado("open");

## What is worth finding out before deciding?

In [ ]:
print(f"Perfect information on everything is worth {r.evpi():,.0f}")
for v in r.value_of_information():
    print(f"  {v.variable:<14} {v.value:>8,.0f}  ({v.share_of_evpi:.0%})")

## What if sales were known?

In [ ]:
sweep = d.sweep("monthly_sales", np.linspace(8000, 18000, 11))
print("Opening breaks even at monthly sales of", round(sweep.breakeven("open", "dont_open")))
sweep.plot();

## Correlated inputs and conditional results

In [ ]:
d.correlate("monthly_sales", "margin", 0.5)
correlated = d.run()
print(f"P(loss) independent: {r.prob_loss('open'):.1%}")
print(f"P(loss) correlated:  {correlated.prob_loss('open'):.1%}")
correlated.given("monthly_sales > 14000")

## Share it

In [ ]:
r.to_html("coffee_shop_report.html")
r.to_dataframe("summary")